# Direct Multi-Step — Random Forest baseline (10 model)


In [34]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    root_mean_squared_error,
)

In [35]:
def get_file_path(filename):
    current_dir = Path.cwd()
    for search_root in [current_dir] + list(current_dir.parents):
        for file in search_root.rglob(filename):
            if file.is_file():
                return file
    raise FileNotFoundError(f"Không tìm thấy file {filename}!")


train_df = pd.read_csv(get_file_path("train_final.csv"))
val_df = pd.read_csv(get_file_path("val_set.csv"))

for df in (train_df, val_df):
    df["Date"] = pd.to_datetime(df["Date"])

print("train :", train_df.shape, "|", train_df["Date"].min().date(), "->", train_df["Date"].max().date())
print("val   :", val_df.shape, "|", val_df["Date"].min().date(), "->", val_df["Date"].max().date())

train : (2700, 24) | 2011-02-04 -> 2012-03-23
val   : (1395, 24) | 2012-03-30 -> 2012-10-26


In [56]:
HORIZON = 10


def add_targets(df, horizon=HORIZON):
    d = df.sort_values(["Store", "Date"]).reset_index(drop=True).copy()
    for h in range(1, horizon + 1):
        d[f"target_t+{h}"] = d.groupby("Store")["Weekly_Sales"].shift(-h)
    return d


train_set = add_targets(train_df)
val_set = add_targets(val_df)

target_cols = [f"target_t+{h}" for h in range(1, HORIZON + 1)]
#visual trên đúng store 1, bảng target)
display(train_set.loc[train_set["Store"] == 1, ["Store", "Date", "Weekly_Sales"] + target_cols[:4]].head(8))
print('dataset đã đủ 10 target cho mô hình khi train')
train_set

,Store,Date,Weekly_Sales,target_t+1,target_t+2,target_t+3,target_t+4
0,1,2011-02-04,1606629.58,1649614.93,1686842.78,1456800.28,1636263.41
1,1,2011-02-11,1649614.93,1686842.78,1456800.28,1636263.41,1553191.63
2,1,2011-02-18,1686842.78,1456800.28,1636263.41,1553191.63,1576818.06
3,1,2011-02-25,1456800.28,1636263.41,1553191.63,1576818.06,1541102.38
4,1,2011-03-04,1636263.41,1553191.63,1576818.06,1541102.38,1495064.75
5,1,2011-03-11,1553191.63,1576818.06,1541102.38,1495064.75,1614259.35
6,1,2011-03-18,1576818.06,1541102.38,1495064.75,1614259.35,1559889.00
7,1,2011-03-25,1541102.38,1495064.75,1614259.35,1559889.00,1564819.81


dataset đã đủ 10 target cho mô hình khi train


,Store,Date,IsHoliday,Weekly_Sales,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,...,target_t+1,target_t+2,target_t+3,target_t+4,target_t+5,target_t+6,target_t+7,target_t+8,target_t+9,target_t+10
0,1,2011-02-04,0,1606629.58,A,151315,42.27,2.989,0.00,0.00,...,1649614.93,1686842.78,1456800.28,1636263.41,1553191.63,1576818.06,1541102.38,1495064.75,1614259.35,1559889.00
1,1,2011-02-11,1,1649614.93,A,151315,36.39,3.022,0.00,0.00,...,1686842.78,1456800.28,1636263.41,1553191.63,1576818.06,1541102.38,1495064.75,1614259.35,1559889.00,1564819.81
2,1,2011-02-18,0,1686842.78,A,151315,57.36,3.045,0.00,0.00,...,1456800.28,1636263.41,1553191.63,1576818.06,1541102.38,1495064.75,1614259.35,1559889.00,1564819.81,1455090.69
3,1,2011-02-25,0,1456800.28,A,151315,62.90,3.065,0.00,0.00,...,1636263.41,1553191.63,1576818.06,1541102.38,1495064.75,1614259.35,1559889.00,1564819.81,1455090.69,1629391.28
4,1,2011-03-04,0,1636263.41,A,151315,59.58,3.288,0.00,0.00,...,1553191.63,1576818.06,1541102.38,1495064.75,1614259.35,1559889.00,1564819.81,1455090.69,1629391.28,1604775.58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2695,45,2012-02-24,0,753060.78,B,118221,42.86,3.739,9006.21,5786.94,...,782796.01,776968.87,788340.23,791835.37,NaN,NaN,NaN,NaN,NaN,NaN
2696,45,2012-03-02,0,782796.01,B,118221,41.55,3.816,22832.38,2515.25,...,776968.87,788340.23,791835.37,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2697,45,2012-03-09,0,776968.87,B,118221,45.52,3.848,11139.34,678.08,...,788340.23,791835.37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2698,45,2012-03-16,0,788340.23,B,118221,50.56,3.862,5811.44,375.70,...,791835.37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [57]:
drop_cols = ["Date", "Weekly_Sales", "Type"] + target_cols
feature_cols = [c for c in train_set.columns if c not in drop_cols]
# đã thử drop store, year thử nhưng kết quả vẫn vậy 
print(f"{len(feature_cols)} cột feature:")
print(feature_cols)

obj_cols = [c for c in feature_cols if train_set[c].dtype == object]
assert not obj_cols, f"Còn cột chuỗi trong feature: {obj_cols}"

21 cột feature:
['Store', 'IsHoliday', 'Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Lag_1', 'Lag_4', 'Lag_12', 'Lag_52', 'Rolling_Mean_4w', 'Year', 'Month', 'WeekOfYear', 'Type_encoded']


In [58]:
rf_models = {}
metrics = []
pred_frames = []

for h in range(1, HORIZON + 1):
    target = f"target_t+{h}"
    train_clean = train_set.dropna(subset=[target])
    val_clean = val_set.dropna(subset=[target])

    X_train, y_train = train_clean[feature_cols], train_clean[target]
    X_val, y_val = val_clean[feature_cols], val_clean[target]

    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42,
    )
    rf.fit(X_train, y_train)
    rf_models[h] = rf

    preds = rf.predict(X_val)

    metrics.append(
        {
            "horizon": h,
            "n_train": len(train_clean),
            "n_val": len(val_clean),
            "MAE": mean_absolute_error(y_val, preds),
            "RMSE": root_mean_squared_error(y_val, preds),
            "R2": r2_score(y_val, preds),
            "WAPE": np.abs(y_val - preds).sum() / np.abs(y_val).sum(),
        }
    )

    frame = val_clean[["Store", "Date"]].copy()
    frame["horizon"] = h
    frame["target_Date"] = frame["Date"] + pd.to_timedelta(h * 7, unit="D")
    frame["y_true"] = y_val.to_numpy()
    frame["y_pred"] = preds
    pred_frames.append(frame)

    m = metrics[-1]
    print(
        f"t+{h:02d}"
        f" | RMSE {m['RMSE']:12,.0f} | WAPE {m['WAPE']:6.2%} | MAE {m['MAE']:12,.0f}"
    )

predictions = pd.concat(pred_frames, ignore_index=True)
print(f"\nBảng dự đoán: {predictions.shape}")

t+01 | RMSE       88,928 | WAPE  5.69% | MAE       59,311
t+02 | RMSE      106,726 | WAPE  6.88% | MAE       71,341
t+03 | RMSE       78,007 | WAPE  4.93% | MAE       51,114
t+04 | RMSE       79,977 | WAPE  5.21% | MAE       54,108
t+05 | RMSE       79,274 | WAPE  5.14% | MAE       53,504
t+06 | RMSE       88,348 | WAPE  5.91% | MAE       61,551
t+07 | RMSE       85,910 | WAPE  5.45% | MAE       56,749
t+08 | RMSE       88,236 | WAPE  5.66% | MAE       58,906
t+09 | RMSE       95,329 | WAPE  6.31% | MAE       65,602
t+10 | RMSE       93,814 | WAPE  6.12% | MAE       63,550

Bảng dự đoán: (11475, 6)


In [59]:
rf_params = dict(n_estimators=200, max_depth=12, min_samples_split=5, min_samples_leaf=2, n_jobs=-1, random_state=42)

rf_models, metrics, pred_frames = {}, [], []

for h in range(1, HORIZON + 1):
    target = f"target_t+{h}"
    train_c = train_set.dropna(subset=[target])
    val_c   = val_set.dropna(subset=[target])
    X_train, y_train = train_c[feature_cols], train_c[target]
    X_val,   y_val   = val_c[feature_cols],   val_c[target]
    rf = RandomForestRegressor(**rf_params).fit(X_train, y_train)
    preds = rf.predict(X_val)
    rf_models[h] = rf


    pred_frames.append(pd.DataFrame({
        "Store": val_c["Store"],
        "Date": val_c["Date"],
        "target_Date": val_c["Date"] + pd.Timedelta(days=h*7),
        "horizon": h,
        "y_true": y_val,
        "y_pred": preds
    }))
    mae  = mean_absolute_error(y_val, preds)
    rmse = root_mean_squared_error(y_val, preds)
    wape = np.abs(y_val - preds).sum() / y_val.sum()
    metrics.append({"horizon": f"t+{h:02d}", "n_train": len(train_c), "n_val": len(val_c), "RMSE": rmse, "WAPE": wape, "MAE": mae})
    print(f"t+{h:02d} | RMSE {rmse:10,.0f} | WAPE {wape:6.2%} | MAE {mae:10,.0f}")

predictions = pd.concat(pred_frames, ignore_index=True)
metrics_df  = pd.DataFrame(metrics).set_index("horizon")

t+01 | RMSE     88,928 | WAPE  5.69% | MAE     59,311
t+02 | RMSE    106,726 | WAPE  6.88% | MAE     71,341
t+03 | RMSE     78,007 | WAPE  4.93% | MAE     51,114
t+04 | RMSE     79,977 | WAPE  5.21% | MAE     54,108
t+05 | RMSE     79,274 | WAPE  5.14% | MAE     53,504
t+06 | RMSE     88,348 | WAPE  5.91% | MAE     61,551
t+07 | RMSE     85,910 | WAPE  5.45% | MAE     56,749
t+08 | RMSE     88,236 | WAPE  5.66% | MAE     58,906
t+09 | RMSE     95,329 | WAPE  6.31% | MAE     65,602
t+10 | RMSE     93,814 | WAPE  6.12% | MAE     63,550


In [60]:
results = pd.DataFrame(metrics).set_index("horizon")
results_view = results.copy()
results_view["WAPE"] = (results_view["WAPE"] * 100).round(2).astype(str) + "%"

display(results_view.round({"MAE": 0, "RMSE": 0, "R2": 4}))

,n_train,n_val,RMSE,WAPE,MAE
horizon,,,,,
t+01,2655,1350,88928.0,5.69%,59311.0
t+02,2610,1305,106726.0,6.88%,71341.0
t+03,2565,1260,78007.0,4.93%,51114.0
t+04,2520,1215,79977.0,5.21%,54108.0
t+05,2475,1170,79274.0,5.14%,53504.0
t+06,2430,1125,88348.0,5.91%,61551.0
t+07,2385,1080,85910.0,5.45%,56749.0
t+08,2340,1035,88236.0,5.66%,58906.0
t+09,2295,990,95329.0,6.31%,65602.0


trc drop store, year, 59.311

### So sánh công bằng giữa các horizon

Bảng trên có một chỗ đọc dễ nhầm: mỗi horizon được chấm trên một tập val **khác nhau**. h=1 có 30 tuần gốc, h=10 chỉ còn 21 — chín tuần gốc chênh nhau (cuối tháng 8 → giữa tháng 10) chỉ xuất hiện ở các horizon ngắn. Nên đường sai số tăng theo `h` trộn hai thứ: horizon xa hơn (cái ta muốn đo) và tập chấm điểm đổi (nhiễu).

Cell dưới lọc cả 10 horizon về chung 21 tuần gốc đầu — lúc này chênh lệch giữa các `h` mới thuần là do khoảng cách dự báo. Việc lọc chỉ diễn ra ở khâu báo cáo, không đụng gì tới lúc `fit`: model nào cũng vẫn được train trên toàn bộ dữ liệu nó có.

In [43]:
all_dates = sorted(val_set["Date"].unique())
common_dates = all_dates[:-HORIZON] # chỉ lấy 21 tuần đầu để đánh giá sai số cho công bằng trên cùng 1 số lượng mẫu
common_df = predictions[predictions["Date"].isin(common_dates)]

records = []
for h in range(1, HORIZON + 1):
    sub = common_df[common_df["horizon"] == h]
    y_true, y_pred = sub["y_true"], sub["y_pred"]
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    wape = np.abs(y_true - y_pred).sum() / np.abs(y_true).sum() * 100

    records.append({
        "Horizon": f"t+{h:02d}",
        "Số dòng": len(sub),
        "RMSE": f"{rmse:,.0f}",
        "WAPE": f"{wape:.2f}%",
        "MAE": f"{mae:,.0f}"
    })

summary_df = pd.DataFrame(records).set_index("Horizon")

print(f"Đánh giá trên {len(common_dates)} tuần gốc: {pd.to_datetime(common_dates[0]).strftime('%Y-%m-%d')} -> {pd.to_datetime(common_dates[-1]).strftime('%Y-%m-%d')}\n")
display(summary_df)

Đánh giá trên 21 tuần gốc: 2012-03-30 -> 2012-08-17



,Số dòng,RMSE,WAPE,MAE
Horizon,,,,
t+01,945,"88,400",5.55%,"58,498"
t+02,945,"93,499",6.03%,"63,158"
t+03,945,"81,119",5.00%,"52,393"
t+04,945,"82,028",5.24%,"54,933"
t+05,945,"81,153",5.09%,"53,328"
t+06,945,"86,590",5.68%,"59,357"
t+07,945,"89,760",5.69%,"59,438"
t+08,945,"90,977",5.85%,"61,090"
t+09,945,"96,594",6.39%,"66,553"


# dùng lag 52 

In [63]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

all_dates = sorted(val_set["Date"].unique())
common_dates = all_dates[:-HORIZON]
common_df = predictions[predictions["Date"].isin(common_dates)].copy()

# 2. Ghép 1 dòng để lấy Lag_52 của tuần đích (target_Date)
common_df = common_df.merge(
    val_set[["Store", "Date", "Lag_52"]],
    left_on=["Store", "target_Date"],
    right_on=["Store", "Date"],
    suffixes=("", "_lookup")
)
print(common_df.head())
# 3. Vòng lặp tính toán ngắn gọn y hệt đoạn trên
records = []
for h in range(1, HORIZON + 1):
    sub = common_df[common_df["horizon"] == h]
    y_true, y_pred_lag52 = sub["y_true"], sub["Lag_52"]

    mae  = mean_absolute_error(y_true, y_pred_lag52)
    rmse = root_mean_squared_error(y_true, y_pred_lag52)
    wape = np.abs(y_true - y_pred_lag52).sum() / y_true.sum() * 100

    records.append({
        "Horizon": f"t+{h:02d}",
        "Số dòng": len(sub),
        "RMSE": f"{rmse:,.0f}",
        "WAPE": f"{wape:.2f}%",
        "MAE": f"{mae:,.0f}"
    })

df_lag52_summary = pd.DataFrame(records).set_index("Horizon")
display(df_lag52_summary)

   Store       Date target_Date  horizon      y_true        y_pred  \
0      1 2012-03-30  2012-04-06        1  1899676.88  1.579107e+06   
1      1 2012-04-06  2012-04-13        1  1621031.70  1.635030e+06   
2      1 2012-04-13  2012-04-20        1  1521577.87  1.610499e+06   
3      1 2012-04-20  2012-04-27        1  1468928.37  1.614176e+06   
4      1 2012-04-27  2012-05-04        1  1684519.99  1.633548e+06   

  Date_lookup      Lag_52  
0  2012-04-06  1614259.35  
1  2012-04-13  1559889.00  
2  2012-04-20  1564819.81  
3  2012-04-27  1455090.69  
4  2012-05-04  1629391.28  


,Số dòng,RMSE,WAPE,MAE
Horizon,,,,
t+01,945,"93,700",5.89%,"62,021"
t+02,945,"85,718",5.48%,"57,385"
t+03,945,"86,389",5.50%,"57,675"
t+04,945,"82,518",5.25%,"55,025"
t+05,945,"83,153",5.28%,"55,379"
t+06,945,"83,812",5.33%,"55,702"
t+07,945,"84,470",5.31%,"55,560"
t+08,945,"83,599",5.26%,"54,900"
t+09,945,"83,446",5.18%,"54,000"


In [ ]:
MODEL_NAME = "RF_direct"

# kiem tra merge khong lam mat dong nao
assert common_df["Lag_52"].notna().all(), "thieu Lag_52 cho mot so tuan dich"

cmp = []
for h in range(1, HORIZON + 1):
    s = common_df[common_df["horizon"] == h]
    w_rf  = np.abs(s["y_true"] - s["y_pred"]).sum()  / np.abs(s["y_true"]).sum()
    w_n52 = np.abs(s["y_true"] - s["Lag_52"]).sum()  / np.abs(s["y_true"]).sum()
    cmp.append({"model": MODEL_NAME, "horizon": h, "n": len(s),
                "WAPE": w_rf, "WAPE_naive52": w_n52, "skill": 1 - w_rf / w_n52})

cmp = pd.DataFrame(cmp)

view = cmp.set_index("horizon")[["n", "WAPE", "WAPE_naive52", "skill"]].copy()
for c in ("WAPE", "WAPE_naive52", "skill"):
    view[c] = (view[c] * 100).round(2).astype(str) + "%"
display(view)

print(f"{MODEL_NAME}: {cmp['WAPE'].mean():.2%} | Naive-52: {cmp['WAPE_naive52'].mean():.2%}"
      f" | skill trung binh {cmp['skill'].mean():+.1%}  (duong = thang baseline)")

# de danh so voi DTree / recursive sau:
# cmp.to_csv(f"cmp_{MODEL_NAME}.csv", index=False)